## 02 - Silver Transform: orders

Limpeza da tabela `orders` (camada Bronze → Silver)

In [0]:
%python
from pyspark.sql.functions import col, to_timestamp

# Leitura da camada Bronze
orders_bronze = spark.table("olist_project.bronze.orders")

# Colunas de data que precisam virar timestamp
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

# Colunas que NUNCA deveriam ser nulas (chaves) - filtramos por elas
required_columns = ["order_id", "customer_id"]

orders_silver = orders_bronze.dropDuplicates(["order_id"])

for date_col in date_columns:
    orders_silver = orders_silver.withColumn(date_col, to_timestamp(col(date_col)))

for required_col in required_columns:
    orders_silver = orders_silver.filter(col(required_col).isNotNull())

# Escrita como Delta Table
(
    orders_silver.write.format("delta")
    .mode("overwrite")
    .saveAsTable("olist_project.silver.orders")
)

print(f"orders_silver: {orders_silver.count()} linhas")

### Comentário:

**1. Leitura da camada bronze**

`orders_bronze = spark.table("olist_project.bronze.orders")`

A camada Silver nunca volta no arquivo bruto, ela sempre parte da `Bronze`. Isso é o princípio da arquitetura medalhão: cada camada só conversa com a camada imediatamente anterior. Se um dia você trocar a fonte de dados (outro sistema, outro formato de arquivo), só o notebook Bronze muda, Silver e Gold continuam funcionando do mesmo jeito, porque eles não sabem (nem precisam saber) de onde o dado bruto veio.

Mais comentários leia a docuemntação `docs/tutorials/how_to_do`.

**2. Por que dropDuplicates**`(["order_id"])` **e não** `dropDuplicates()` sem argumento

`orders_silver = orders_bronze.dropDuplicates(["order_id"])`

`dropDuplicates()` sem argumento remove linhas totalmente idênticas em todas as colunas, isso não protege contra o caso real que a gente quer evitar: duas linhas com o mesmo `order_id`, mas com algum campo levemente diferente (ex: um timestamp com formatação diferente). Especificar` ["order_id"]` garante o critério de aceite de verdade: zero duplicatas pela chave, não só duplicatas "perfeitas".

**3. Por que converter as datas com um loop, e não linha por linha**

`for date_col in date_columns:
    orders_silver = orders_silver.withColumn(date_col, to_timestamp(col(date_col)))`

São 5 colunas de data na tabela orders. 

Escrever `.withColumn(...)` 5 vezes manualmente funcionaria, mas o loop deixa explícito que é a mesma operação repetida, e se amanhã aparecer uma 6ª coluna de data, você só adiciona o nome na lista `date_columns`, não escreve uma linha de código nova.

`to_timestamp()` converte a coluna (que veio como string ou tipo inferido incorretamente do CSV) pro tipo `timestamp` de verdade, é isso que satisfaz o **segundo critério de aceite da Issue.**

**4. Por que filtrar nulo só em `order_id` e `customer_id`, não nas datas**

`required_columns = ["order_id", "customer_id"]

for required_col in required_columns:
    orders_silver = orders_silver.filter(col(required_col).isNotNull())`

Atenção aqui:

No dicionário de dados já documentamos que datas de entrega `nulas` são esperadas (um pedido cancelado nunca teve `order_delivered_customer_date`, por exemplo, isso não é erro, é o próprio negócio). 

Se a gente filtrasse `nulo` em todas as colunas, incluindo as de data, perderíamos pedidos cancelados inteiros da tabela `Silver`, um erro de lógica que passaria despercebido sem essa documentação prévia.

Já `order_id` e `customer_id` são chaves, um pedido sem `order_id` não é um pedido válido, é lixo de dado (erro de ingestão, linha corrompida). Faz sentido descartar essas linhas.

Isso é a diferença entre limpeza de dado feita "no automático" (que quebraria silenciosamente) e limpeza feita com entendimento do `domínio`.

**5.Por que `mode("overwrite")` de novo**

Mesma razão do notebook `Bronze: idempotência`. Rodar esse notebook 5 vezes deve dar o mesmo resultado que rodar 1 vez, sem duplicar nem acumular lixo.


## Validação

Essa query prova o critério de aceite "zero duplicatas".
É importante validar explicitamente

In [0]:
SELECT order_id, COUNT(*) as qtd
FROM olist_project.silver.orders
GROUP BY order_id
HAVING COUNT(*) > 1;

## Colunas de data devem estar com tipo timestamp

In [0]:
DESCRIBE olist_project.silver.orders;

**DESCRIBE**:

Mostra o schema da tabela, incluindo o tipo de cada coluna, é como a gente confirma visualmente que as 5 colunas de data realmente viraram `timestamp`, satisfazendo o segundo critério de aceite.

## 02b - Silver Transform: customers, products, sellers

### 1. Tabela customers

O que faremos:

Pegaremos a tabela `customers` da camada Bronze (como veio, sem tratamento), removemos linhas com `customer_id` duplicado, descartamos linhas sem `customer_id` (dado inválido), e salvamos o resultado como uma nova tabela `Delta` na camada Silver. O print final serve só pra confirmar visualmente quantas linhas sobreviveram depois da limpeza.

In [0]:
%python
from pyspark.sql.functions import col, to_timestamp

In [0]:
%python
customers_bronze = spark.table("olist_project.bronze.customers")

customers_silver = (
    customers_bronze
    .dropDuplicates(["customer_id"]) # remove linhas duplicadas, mas considerando apenas a coluna customer_id como critério
    .filter(col("customer_id").isNotNull()) # descarta qualquer linha onde customer_id seja nulo /.isNotNull() é a condição booleana
)

(
    customers_silver.write.format("delta")
    .mode("overwrite")
    .saveAsTable("olist_project.silver.customers")
)

print(f"customers_silver: {customers_silver.count()} linhas")


**Resumo do fluxo estudado**

customers (Bronze, com possíveis duplicatas e nulos)

   → remove duplicatas por `customer_id`

   → remove linhas sem `customer_id`

   → customers (Silver, chave única e garantida)

### 2. Tabela products

O que faremos:

Vamos lêer a tabela `products` da camada Bronze, exatamente como veio da ingestão. Sem nenhuma limpeza ainda.
Vamos realizar duas operações encadeadas:

Duas operações encadeadas:

.dropDuplicates(["product_id"]) → remove linhas com product_id repetido, mantendo só uma ocorrência de cada produto

.filter(col("`product_id`").isNotNull()) → descarta linhas onde `product_id` é nulo (produto sem identificador não é um registro válido)

Salvaremos o resultado como tabela Delta em `olist_project.silver.products`, sobrescrevendo se já existir (idempotência).

In [0]:
%python
products_bronze = spark.table("olist_project.bronze.products")

products_silver = (
    products_bronze
    .dropDuplicates(["product_id"])
    .filter(col("product_id").isNotNull())
)

(
    products_silver.write.format("delta")
    .mode("overwrite")
    .saveAsTable("olist_project.silver.products")
)

print(f"products_silver: {products_silver.count()} linhas")


## 3. Tabela sellers

O que faremos:

- Lê sellers da Bronze
- Remover duplicatas por `seller_id` (garante chave única)
- Descartar linhas com `seller_id` nulo (dado inválido)
- Salvar como Delta em `olist_project.silver.sellers`, sobrescrevendo se já existir
- Imprimir a contagem final como confirmação


In [0]:
%python
# Lê sellers da Bronze
sellers_bronze = spark.table("olist_project.bronze.sellers")

# Remove duplicatas por seller_id
# Descarta linhas com seller_id nulo (dado inválido)
sellers_silver = (
    sellers_bronze
    .dropDuplicates(["seller_id"])
    .filter(col("seller_id").isNotNull())
)
# Salva como Delta em olist_project.silver.sellers, sobrescrevendo se já existir
(
    sellers_silver.write.format("delta")
    .mode("overwrite")
    .saveAsTable("olist_project.silver.sellers")
)

print(f"sellers_silver: {sellers_silver.count()} linhas")